In [1]:
# ## install finrl library
!pip install wrds
!pip install swig
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install finrl



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Abhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Abhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Abhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install gym
!pip install tensorboard



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Abhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Abhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

import gym
from finrl import config
from finrl.agents.stablebaselines3.models import DRLEnsembleAgent
from finrl.meta.preprocessor.preprocessors import FeatureEngineer

from pprint import pprint

import sys
sys.path.append("../FinRL-Library")

import itertools

In [5]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

In [6]:
TRAIN_START_DATE = '2017-01-01'
TRAIN_END_DATE = '2021-10-01'
TEST_START_DATE = '2021-10-01'
TEST_END_DATE = '2023-03-01'

In [7]:
df=pd.read_csv("C:/Users/Abhi/Downloads/TSX Chart Data Sorted By Date.csv")

In [8]:
#Data Exploration
print("First 5 rows of the dataset:")
print(df.head())

First 5 rows of the dataset:
         date      close    high     low    open  volume     tic  day
0  2004-01-01   0.105000   0.105   0.105   0.105       0  DYA.TO    3
1  2004-01-01   1.378000   2.650   2.650   2.650       0  EDT.TO    3
2  2004-01-02   0.489625   0.530   0.530   0.530       0  AAB.TO    4
3  2004-01-02  22.070272  29.850  29.320  29.430  413000  ABX.TO    4
4  2004-01-02   3.484277   7.050   7.050   7.050     100  ACD.TO    4


In [9]:
print(df.isnull().sum())  # Shows the count of missing values per column

date      0
close     0
high      0
low       0
open      0
volume    0
tic       0
day       0
dtype: int64


In [10]:
# Convert 'date' column to datetime if it is not already
df['date'] = pd.to_datetime(df['date']) 

# *Step 1: Data Cleaning*
df = df.sort_values(["date", "tic"], ignore_index=True)

# Factorize the date index
df.index = df.date.factorize()[0]
df

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.10500,0.105000,0.10500,0,DYA.TO,3
0,2004-01-01,1.378000,2.65000,2.650000,2.65000,0,EDT.TO,3
1,2004-01-02,0.489625,0.53000,0.530000,0.53000,0,AAB.TO,4
1,2004-01-02,22.070272,29.85000,29.320000,29.43000,413000,ABX.TO,4
1,2004-01-02,3.484277,7.05000,7.050000,7.05000,100,ACD.TO,4
...,...,...,...,...,...,...,...,...
5279,2024-12-27,21.600000,21.65000,21.219999,21.40000,153600,ELD.TO,4
5279,2024-12-27,0.370000,0.38000,0.360000,0.36000,16900,ELEF.TO,4
5279,2024-12-27,1311.250000,1319.98999,1299.520020,1299.52002,1200,ELF.TO,4
5279,2024-12-27,0.850000,0.88000,0.840000,0.88000,116800,ELO.TO,4


In [11]:

# Drop rows where any of the required columns have NaN values
df = df.dropna(subset=["close", "high", "low", "open", "tic", "day"])

# *Step 3: Add User Defined Features*
df["daily_return"] = df.close.pct_change(1)

In [12]:
df.head()

,date,close,high,low,open,volume,tic,day,daily_return
0,2004-01-01,0.105000,0.105,0.105,0.105,0,DYA.TO,3,NaN
0,2004-01-01,1.378000,2.650,2.650,2.650,0,EDT.TO,3,12.123810
1,2004-01-02,0.489625,0.530,0.530,0.530,0,AAB.TO,4,-0.644684
1,2004-01-02,22.070272,29.850,29.320,29.430,413000,ABX.TO,4,44.075841
1,2004-01-02,3.484277,7.050,7.050,7.050,100,ACD.TO,4,-0.842128


In [13]:
INDICATORS = ['macd',
               'rsi_30',
               'cci_30',
               'dx_30']

In [14]:
TICKERS = ["RY.TO", "TD.TO", "BNS.TO", "ENB.TO", "SHOP.TO"]  # Example TSX tickers

In [15]:
df = df[df['tic'].isin(TICKERS)]
df.shape

(5271, 9)

In [16]:
feature_engineer = FeatureEngineer(use_technical_indicator=True, tech_indicator_list=INDICATORS)


In [17]:
df = feature_engineer.add_technical_indicator(df)

In [18]:
df = feature_engineer.add_turbulence(df)

In [19]:
df["date"] = pd.to_datetime(df["date"])  # Convert main dataset date column

In [20]:
stock_dimension = len(df.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 1, State Space: 7


In [21]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5

}


In [22]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window,
                 validation_window=validation_window,
                 **env_kwargs)

In [23]:


A2C_model_kwargs = {
                    'n_steps': 5,
                    'ent_coef': 0.005,
                    'learning_rate': 0.0007
                    }

TD3_model_kwargs = {"batch_size": 100, "buffer_size": 1000000, "learning_rate": 0.0001}




timesteps_dict = {'a2c' : 10_000,
                 'ppo' : 0,
                 'ddpg' : 0,
                 'sac' : 0,
                 'td3' : 0
                 }

In [24]:
df_summary = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs=A2C_model_kwargs,  # Using A2C
    PPO_model_kwargs={},  # Not using PPO
    DDPG_model_kwargs={},  # Not using DDPG
    SAC_model_kwargs={},  # Not using SAC
    TD3_model_kwargs={},  # Not using TD3
    timesteps_dict=timesteps_dict
)

============Start Ensemble Strategy============
turbulence_threshold:  14.337119252361923
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{'n_steps': 5, 'ent_coef': 0.005, 'learning_rate': 0.0007}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_1
------------------------------------
| time/                 |          |
|    fps                | 316      |
|    iterations         | 100      |
|    time_elapsed       | 1        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -1.47    |
|    explained_variance | 0        |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -0.0518  |
|    reward             | 0.0      |
|    std                | 1.05     |
|    value_loss         | 0.000488 |
------------------------------------
----------------------------------------
| time/                 |              |
|    fps                | 329

In [25]:
df_summary

,Iter,Val Start,Val End,Model Used,A2C Sharpe,PPO Sharpe,DDPG Sharpe,SAC Sharpe,TD3 Sharpe
0,126,2021-10-04 00:00:00,2022-01-05 00:00:00,PPO,0.344848,0.6477,0.215648,0.215648,0.215648
1,189,2022-01-05 00:00:00,2022-04-05 00:00:00,TD3,-0.112719,-0.080389,-0.066235,-0.066235,0.0
2,252,2022-04-05 00:00:00,2022-07-06 00:00:00,DDPG,-0.37277,-0.152435,0.0,-0.37277,-0.37277
3,315,2022-07-06 00:00:00,2022-10-05 00:00:00,A2C,-0.031265,-0.495614,-0.102006,-0.102006,-0.102006
